# Day 67 — Improved Model: Retail Demand Predictor Mini-Project

This notebook covers Random Forest Regressor, Non-Linear Feature Interactions, Multiplicative Economic Dynamics, Hyperparameter Configuration (`n_estimators=150`, `max_depth=12`), and Side-by-Side Artifact Tracking

## Validation goals
- Implement an improved model using `RandomForestRegressor` to capture non-linearities and complex feature interactions in retail demand data
- Configure ensemble hyperparameters (`n_estimators=150`, `max_depth=12`) to balance model capacity and regularization
- Evaluate the improved model using regression metrics (MAE, RMSE, and $R^2$) and quantify performance gains over the linear baseline
- Update the shared `retail_demand_predictions.csv` artifact to store Random Forest predictions side-by-side for comparative analysis

### 1. Why Tree-Based Ensembles Shine in Retail

In retail demand forecasting, relationships are rarely a straight line:
- A price hike from ₦1,000 to ₦2,000 might barely affect demand for essential food items, but the exact same price jump from ₦50,000 to ₦51,000 on electronics could cause sales to plummet. That is non-linearity.
- Demand surges during festive periods (season == 'Festive') interact dynamically with regional purchasing power and historical sales.

Tree-based models like Random Forests naturally partition feature space into rectangular regions, allowing them to capture complex interactions, thresholds, and non-linear behaviors without requiring manual feature engineering.

### 2. Implementing the Improved Model

Let's write a clean script that loads our preprocessed CSV files from disk, trains a `RandomForestRegressor`, evaluates its performance against our test set, and updates our shared predictions CSV so we can easily compare it to our baseline tomorrow on Day 68.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the preprocessed train and test datasets from Day 65
train_df = pd.read_csv("retail_demand_train_processed.csv")
test_df = pd.read_csv("retail_demand_test_processed.csv")

X_train = train_df.drop(columns=['demand_units'])
y_train = train_df['demand_units']

X_test = test_df.drop(columns=['demand_units'])
y_test = test_df['demand_units']

# 2. Instantiate and train the Improved Model (Random Forest Regressor)
# n_estimators builds 150 decision trees; max_depth prevents overfitting
improved_model = RandomForestRegressor(
    n_estimators=150, 
    max_depth=12, 
    random_state=42, 
    n_jobs=-1
)

improved_model.fit(X_train, y_train)

# 3. Generate predictions on the unseen test set
y_pred_rf = improved_model.predict(X_test)

# 4. Compute evaluation metrics
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2 = r2_score(y_test, y_pred_rf)

print("--- Day 67: Improved Model (Random Forest) Results ---")
print(f"Random Forest MAE:  {mae:.2f} units")
print(f"Random Forest RMSE: {rmse:.2f} units")
print(f"Random Forest R²:   {r2:.4f}\n")

# 5. Update our existing predictions CSV to store these predictions side-by-side
predictions_df = pd.read_csv("retail_demand_predictions.csv")
predictions_df['random_forest_pred'] = np.round(y_pred_rf, 2)
predictions_df.to_csv("retail_demand_predictions.csv", index=False)

print("Updated 'retail_demand_predictions.csv' with Random Forest test predictions.")

--- Day 67: Improved Model (Random Forest) Results ---
Random Forest MAE:  9.75 units
Random Forest RMSE: 14.46 units
Random Forest R²:   0.9627

Updated 'retail_demand_predictions.csv' with Random Forest test predictions.


### Guided Question:

Why are tree-based ensemble models like Random Forests inherently better equipped to handle multiplicative economic factors (such as holiday season surges multiplying historical sales) compared to standard Linear Regression?

### Answer:

The fundamental advantage of tree-based ensemble models like Random Forests over standard Linear Regression in handling multiplicative economic factors (such as holiday surges scaling historical sales) stems from how each model structures relationships mathematically.

Standard linear regression assumes additivity, whereas economic demand dynamics are inherently multiplicative and proportional.

**1. The Additive Assumption vs. Proportional Scaling**

Standard Ordinary Least Squares (OLS) Linear Regression models the target variable $y$ as a linear combination of features:$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \epsilon$$

In this formulation, a coefficient ($\beta_1$) represents a constant absolute change in $y$ for a unit increase in $x_1$, regardless of the current baseline value of $y$.

However, economic factors like holiday seasons, macroeconomic inflation, or promotional discounts do not add a fixed absolute number of units (e.g., "+500 items"). Instead, they act as multipliers that scale baseline demand proportionally (e.g., $\times 1.5$ or a $+50\%$ surge):$$\text{Sales} = \text{Baseline Demand} \times \text{Holiday Multiplier} \times \text{Promotional Lift}$$

To capture this in a linear regression, you must manually apply a log transformation to linearize the relationship:$$\ln(\text{Sales}) = \ln(\text{Baseline}) + \ln(\text{Holiday Multiplier}) + \dots$$

If you fail to apply this transformation, a standard linear model will struggle: it will either underpredict high-volume stores during holidays (since a flat additive coefficient cannot scale with volume) and overpredict low-volume stores, or it will yield biased error residuals. Furthermore, log transformations fail if your dataset contains zero-sales periods or negative economic metrics.

**2. Recursive Partitioning and Local Constant Estimation**

Decision trees operate via recursive binary splitting, partitioning the multi-dimensional feature space into orthogonal hyperrectangles (regions or leaves) and predicting a constant value within each region.

Instead of fitting a single global equation across the entire dataset, a Random Forest builds an ensemble of deep trees that approximate complex surfaces through piecewise constant regions. This architecture handles multiplicative factors natively for several reasons:
- Threshold-Based Scaling: A decision tree can easily isolate a condition (e.g., Month == December AND Store_Type == Flagship) and apply a localized split that captures the proportional jump in sales.
- Scale-Invariance to Monotonic Transformations: Tree splits are based on feature ranks rather than absolute magnitudes. Whether a feature is expressed in raw currency, normalized scales, or multiplicative ratios, the tree's splitting logic identifies the optimal split point without requiring manual feature engineering.

**3. Implicit Multi-Way Interactions**

In economic forecasting, multipliers rarely act in isolation. A holiday surge multiplier interacts heavily with other variables:
- A holiday surge has a vastly different absolute impact on a high-traffic urban store compared to a rural outlet.
- The magnitude of a holiday multiplier often depends on inventory availability, local competitor pricing, and regional foot traffic indices.

In a Linear Regression, capturing these complex interactions requires manually engineering and adding interaction terms (e.g., $x_1 \times x_2 \times x_3$). If you omit an interaction term, the model treats the effects as independent and additive.

In a Random Forest, deep decision trees naturally capture high-order interactions through hierarchical branching. Node 1 can split on store size, Node 2 on geographic region, and Node 3 on the holiday indicator. By averaging hundreds of diverse trees, the Random Forest smooths out these recursive splits to approximate smooth, non-linear multiplicative surfaces without requiring you to explicitly define interaction terms or log-transform your target variable.

### Guided Question:

In the script above, what purpose do `n_estimators=150` and `max_depth=12` serve in controlling the behavior and generalization capacity of the Random Forest?

### Answer:

In the script, `n_estimators=150` and `max_depth=12` act as the primary structural levers governing the bias-variance tradeoff of the Random Forest. Together, they dictate how complex the individual trees are permitted to become and how stably the ensemble aggregates their predictions to generalize to unseen test data (`retail_demand_test_processed.csv`).

**1. `n_estimators=150` (Ensemble Scale & Variance Reduction)**

- What it does: This parameter instructs the algorithm to build 150 independent decision trees within the forest, each trained on a bootstrapped subset of the training data with randomized feature selection (bagging).
- Behavior: When generating predictions (y_pred_rf), the model averages the continuous outputs of all 150 trees. Because each tree captures slightly different patterns due to bootstrap sampling and feature sub-sampling, the aggregate ensemble cancels out individual tree errors.
- Generalization Capacity: Unlike single decision trees, adding more trees does not cause overfitting. As n_estimators increases, the variance of the model's predictions drops steadily until it plateaus. Setting it to 150 ensures sufficient statistical averaging for stable, reliable retail demand estimates without excessive computational overhead.

**2. `max_depth=12` (Complexity Control & Regularization)**

- What it does: This parameter caps the maximum number of sequential splits (from root node down to leaf node) that any individual decision tree in the forest is allowed to make.
- Behavior: Unconstrained decision trees (`max_depth=None`) continue splitting until every leaf is pure or contains very few samples. In economic datasets like retail demand, unconstrained trees will aggressively chase and memorize idiosyncratic training noise—such as a single anomalous promotional spike or a weird weather blip—treating noise as a permanent rule.
- Generalization Capacity: By restricting the depth to 12, you enforce a regularization constraint. Each tree is forced to learn broader, high-level hierarchical interactions (e.g., store type $\rightarrow$ region $\rightarrow$ holiday indicator) rather than hyper-specific micro-rules. This controls model complexity, keeps the variance manageable, and preserves out-of-sample accuracy on the unseen test set.

### Guided Question:

Why did we read `retail_demand_predictions.csv` and append random_forest_pred to it rather than creating an entirely separate CSV file?

### Answer:

Appending the new predictions directly to `retail_demand_predictions.csv` instead of creating a separate file comes down to centralized model comparison, workspace hygiene, and downstream pipeline efficiency.

Here is why structuring your evaluation artifact this way is standard practice in iterative machine learning workflows:

**1. Direct Side-by-Side Benchmarking and Error Analysis**

In an iterative modeling process (moving from a baseline model to an improved Random Forest, and eventually to gradient boosting or neural networks), you need to compare how predictions evolve.
- By housing predictions side-by-side in a single DataFrame alongside actual demand (`y_test`), you can instantly compute comparative metrics, plot residual errors on the exact same scale, and check where the Random Forest outperformed the baseline.
- If you saved every iteration as a separate file (`baseline_preds.csv`, `rf_preds.csv,` `xgb_preds.csv`), your evaluation scripts would require messy multi-file merges just to generate a comparison table or plot.

**2. Streamlining Ensembling and Blending**

Later in model development, you may want to build an ensemble blend (e.g., combining 60% Random Forest predictions with 40% baseline predictions). Having all predictions aligned row-by-row in a single CSV file makes ensemble calculations a simple vector operation:

`predictions_df['ensemble_pred'] = (
    0.6 * predictions_df['random_forest_pred'] + 
    0.4 * predictions_df['baseline_pred']
)`

**3. Workspace Hygiene and Reproducibility**

As an ML project progresses across dozens of development days, creating a new CSV file for every minor tweak or model variation leads to severe file clutter. Maintaining a single, authoritative predictions scorecard ensures that your test harness remains clean, predictable, and easy to inspect as a unified ledger of model performance.